In [1]:
import data.breathe_data as bd
import data.helpers as dh
import src.models.helpers as mh
from plotly.subplots import make_subplots
import datetime
import data.cfr_data_19_23 as cfrd
import pandas as pd
import cfr.cfr_viz_helpers as vh
import plotly.express as px
import plotly.graph_objs as go
from scipy.stats import spearmanr, permutation_test

In [4]:
df = bd.load_meas_from_excel(
    # "infer_all_19_data_with_best_FEV1",
    "ppfev1st_ft_bFEV1_2016-19_IV_2019-21_assoc",
    study_folder="CFR",
    str_cols_to_arrays=[
        # "Airway resistance (%)",
        # "P(HFEV1|FEF2575, bFEV1, FEV1)",
        "P(HFEV1|bFEV1)",
        "P(HFEV1|FEV1)",
    ],
    use_csv=True,
    bypass_sanity_checks=True,
)

In [107]:
diff_col = "ppFEV1FT - ppFEV1ST"

dftmp = df.copy()

# Filter percentile of certain population
prctile = 90
for prctile in [50, 60, 70, 80, 90]:
    t = dftmp[diff_col].abs().quantile(prctile / 100)
    print(f"{prctile}th percentile of absolute difference: {t}")

    dftmp = dftmp[dftmp[diff_col].abs() > t]

    # Filter by severity level
    dftmp.loc[dftmp["FEV1%PredST"] >= 70, "severity"] = "mild"
    dftmp.loc[(dftmp["FEV1%PredST"] >= 40) & (dftmp["FEV1%PredST"] < 70), "severity"] = (
        "moderate"
    )
    dftmp.loc[dftmp["FEV1%PredST"] < 40, "severity"] = "severe"

    fig = make_subplots(rows=3, cols=1, shared_xaxes=True)

    for i, severity in enumerate(["mild", "moderate", "severe"], start=1):
        dftmp_severity = dftmp[dftmp["severity"] == severity]
        fig.add_trace(
            go.Scatter(
                x=dftmp_severity[diff_col], y=dftmp_severity["IV days"], mode="markers", marker=dict(size=3, opacity=0.8)
            ),
            row=i,
            col=1,
        )
        fig.update_yaxes(title="IV days", range=[-5, dftmp["IV days"].max()*1.1], row=i, col=1)
    fig.update_xaxes(title=f"{diff_col}", row=3, col=1)
        

    title = f"bFEV1_2016_19_IVdays_2019-21_{prctile}th_prctile"
    fig.update_layout(
        height=600, width=800, title=title,
    )

    fig.write_image(dh.get_path_to_main() + f"PlotsCFR/Viz diff vs IV days/{title}.pdf")

##########################################################################################
    def labels_from_bins(bins):
        labels = [f"< {bins[1]}"]
        for i in range(1, len(bins) - 2):
            labels.append(f"[{bins[i]}, {bins[i+1]})")
        labels.append(f">= {bins[-2]}")
        return labels


    fig = make_subplots(3, 1)

    row = 0
    for severity in ["mild", "moderate", "severe"]:
        row += 1
        dftmp2 = dftmp[dftmp["severity"] == severity].copy()
        bins = [-1000, -20, -11, -9, -7, -5, -3, -1, 0]
        labels = labels_from_bins(bins)
        dftmp2["diff_bin"] = pd.cut(dftmp2[diff_col], bins=bins, labels=labels)

        grouped = (
            dftmp2.groupby("diff_bin", observed=False)
            .agg(
                iv_mean=("IV days", "mean"),
                iv_std=("IV days", "std"),
                diff_mean=(diff_col, "mean"),
                count=(diff_col, "size"),
            )
            .reset_index()
        )

        fig.add_trace(
            go.Bar(
                x=grouped["diff_bin"].astype(str),
                y=grouped["iv_mean"],
                # mode="markers+text",
                error_y=dict(type="data", array=grouped["iv_std"].tolist(), visible=True),
                text=grouped["count"].astype(int),
                # textposition="top center",
                name=severity,
            ),
            row=row,
            col=1,
        )
        fig.update_yaxes(title="IV days (mean ± SD)", row=row, col=1)
    fig.update_xaxes(title=f"{diff_col} binned", row=1, col=1)


    title = f"bFEV1_2016_19_IVdays_2019-21_binned_{prctile}th_prctile"

    fig.update_layout(
        xaxis_title=diff_col,
        title=title,
        height=800,
    )
    fig.write_image(dh.get_path_to_main() + f"PlotsCFR/Viz diff vs IV days/{title}.pdf")

50th percentile of absolute difference: 0.457945218826417
60th percentile of absolute difference: 2.781535403334735
70th percentile of absolute difference: 5.978852663315695
80th percentile of absolute difference: 12.69587666933772
90th percentile of absolute difference: 25.269813601145305
